[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/csg_raytracer/csg_raytracer.ipynb)

# Solids Cut from Quadrics, Ray Traced

A ruled hyperboloid cut off by two planes, a bowl hollowed out and cut open, and a mirror sphere stand on a checkered floor, lit by a lamp that casts their shadows, while a camera circles them. Every solid is the intersection of a few quadric solids, and the floor and the planes that cut are quadrics too, so one ray tracer shows all of them.

A solid quadric is a map from points to planes, `Plane <- Point`, whose pairing with a point, `quadric(point) & point`, is zero on its surface and negative inside, and at a point of the surface the plane it gives is the tangent plane. A half-space is the product of its plane's pairing with the plane at infinity's, `plane * (mv.w & Point)` symmetrized; negating a quadric swaps inside and outside, which is how the bowl loses its inside. Along a ray the pairing is a quadratic in the distance, and the surface a ray sees first is a root of one solid's quadric that lies inside all its others. The tracer runs on the GPU, in a shader whose expressions are numga's.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import stack
from examples.animation import save_figure, show_animation
from examples.quadrics.csg_raytracer import render
from examples.quadrics.csg_raytracer.core import (
    Motor, Plane, Point, Quadric, ellipsoid, ga, evaluate, half_space, hyperboloid, mv, origin, orbit, placed, ruling_lines, translator,
)
from examples.wgsl import Shader, image_figure, supersampled

## 1. A ruled hyperboloid

A hyperboloid of one sheet is an ellipsoid with the pairing along its axis subtracted, and through every point of its waist pass two straight lines lying in its surface. Turning the waist point and both directions about z together gives them all.

In [ ]:
waist_axes = np.array([0.55, 0.55, 0.85])
waist, rising, falling = ruling_lines(waist_axes, 12)                    # [rulings] Point, twice [rulings] Point
reach = np.linspace(-2, 2, 5)[:, None]
along_rulings = evaluate(hyperboloid(waist_axes), waist + rising * reach)   # [reaches, rulings] Scalar

In [ ]:
with np.printoptions(precision=2, suppress=True):
    print("the hyperboloid along its rulings:", along_rulings.to_array().max())

## 2. Solids as intersections

Each solid is a list of quadrics, all placed by one motor, `motor >> quadric(motor << Point)`: each point moved back, measured, and its plane moved forward. The shader reads every solid's range of quadrics, its colour, how much it mirrors, and two planes whose unit steps lay a checker over it.

In [ ]:
lift = translator(mv.direction([-1.2, 0.4, 0.85])) * (mv.xz * (-np.radians(12.0) / 2)).exp()          # [] Motor
turn_bowl = (translator(mv.direction([1.1, -0.2, 0.75]))
             * (mv.yz * (-np.radians(35.0) / 2)).exp() * (mv.xz * (-np.radians(15.0) / 2)).exp())        # [] Motor
cut = mv.y * 0.15 + mv.z * 0.98 - mv.w * 0.1
height = waist_axes[2]
shapes = [
    # The hyperboloid between the planes a half-height above and below its waist.
    [placed(hyperboloid(waist_axes), lift), placed(half_space(mv.z - mv.w * height), lift),
     placed(half_space(-mv.z - mv.w * height), lift)],
    # The bowl: inside the outer ellipsoid, outside the inner one, below the cut.
    [placed(ellipsoid(np.full(3, 0.75)), turn_bowl), -placed(ellipsoid(np.full(3, 0.6)), turn_bowl),
     placed(half_space(cut / (cut | cut).square_root()), turn_bowl)],
    [placed(ellipsoid(np.full(3, 0.55)), translator(mv.direction([0.0, 0.1, 0.55])))],
    # The floor: everything below the plane z = 0.
    [half_space(mv.z).cast(Quadric)],
]
counts = np.array([len(shape) for shape in shapes])
ends = np.cumsum(counts)
quadrics = stack([quadric for shape in shapes for quadric in shape])                     # [quadrics] Quadric
# Stripes crossing the hyperboloid's rulings, and tiles on the floor.
stripes, tiles = 8.0, 1.4
solids = dict(first=ends - counts, end=ends,
              colour=np.array([[0.84, 0.48, 0.24], [0.15, 0.45, 0.55], [0.95, 0.95, 0.96], [0.88, 0.87, 0.85]]),
              reflectivity=np.array([0.0, 0.0, 0.88, 0.0]),
              across=stack([lift >> (mv.x + mv.z) * stripes, 0 * mv.x, 0 * mv.x, mv.x * tiles]),
              along=stack([lift >> (mv.y - mv.z) * stripes, 0 * mv.y, 0 * mv.y, mv.y * tiles]))

## 3. The shader

Each invocation follows one camera ray. Along it a quadric is a quadratic in the distance, `a t^2 + 2 b t + c`, its coefficients the quadric's pairings of the ray's origin and direction; a half-space has no square term, and the stable form of the roots gives its one root without a case of its own. The tangent plane lights the surface by its pairing with the lamp's direction, `normal & lamp`, and reflects the view ray, `normal >> direction`, for the glint and for the mirror's second look.

In [ ]:
source = supersampled() + """
// A solid: the intersection of its quadrics, `quadrics[first]` up to `quadrics[end]`, each negative
// inside it; its colour, the share of light it reflects as a mirror, and two planes whose unit steps
// lay a checker over it.
struct Solid {
    first: u32,
    end: u32,
    colour: vec3<f32>,
    reflectivity: f32,
    across: Plane,
    along: Plane,
}

var<storage, read> solids: array<Solid>;
var<storage, read> quadrics: array<Quadric>;
var<uniform> camera: Motor;
var<uniform> eye: Point;
var<uniform> lamp: Point;
var<uniform> half_view: f32;
alias Sample = vec3<f32>;

// Distances beyond the scene, and the step a ray takes off a surface before it looks again.
const far = 1e6;
const clearance = 1e-3;
// How far a point may lie outside a quadric and still count as inside, for points on two surfaces.
const tolerance = 1e-4;
// The look: light where the lamp does not reach, a shadow's light, the checker's darkening, and the glint.
const ambient = 0.28;
const shadow = 0.3;
const checker = 0.15;
const shininess = 40.0;
const gloss = 0.3;
const horizon = vec3<f32>(0.80, 0.84, 0.90);
const zenith = vec3<f32>(0.38, 0.52, 0.72);

struct Hit {
    distance: f32,
    tangent: Plane,
    solid: u32,
}

// Whether a point lies inside all of a solid's quadrics but one.
fn inside_others(solid: Solid, own: u32, point: Point) -> bool {
    for (var index = solid.first; index < solid.end; index++) {
        if (index != own && (quadrics[index](point) & point) > tolerance) {
            return false;
        }
    }
    return true;
}

// Where a ray first meets a solid's surface: on one of its quadrics, and inside all the others.
fn first_hit(origin: Point, direction: Point) -> Hit {
    var hit = Hit(far, mv.z, 0u);
    for (var index = 0u; index < arrayLength(&solids); index++) {
        let solid = solids[index];
        for (var own = solid.first; own < solid.end; own++) {
            let quadric = quadrics[own];
            // Along the ray the quadric is a quadratic in the distance, a t^2 + 2 b t + c.
            let a = quadric(direction) & direction;
            let b = ((quadric(direction) & origin) + (quadric(origin) & direction)) / 2;
            let c = quadric(origin) & origin;
            let discriminant = b * b - a * c;
            if (discriminant < 0) {
                continue;
            }
            // Both roots without cancellation; a half-space, with no square term, has only the second.
            let q = -(b + select(-1.0, 1.0, b >= 0) * sqrt(discriminant));
            let roots = vec2<f32>(select(q / a, -1.0, a == 0), c / q);
            for (var which = 0u; which < 2u; which++) {
                let distance = roots[which];
                let point = origin + direction * distance;
                if (distance > clearance && distance < hit.distance && inside_others(solid, own, point)) {
                    // The quadric's plane at a point of its surface is the tangent plane there.
                    hit = Hit(distance, quadric(point), index);
                }
            }
        }
    }
    return hit;
}

struct Sight {
    colour: vec3<f32>,
    point: Point,
    normal: Plane,
    reflectivity: f32,
}

// What a ray sees: the solid it meets, lit by the lamp, shadowed, checkered and glinting; or the sky.
fn sight(origin: Point, direction: Point) -> Sight {
    let hit = first_hit(origin, direction);
    if (hit.distance >= far) {
        return Sight(mix(horizon, zenith, clamp(mv.z & direction, 0.0, 1.0)), origin, mv.z, 0.0);
    }
    let solid = solids[hit.solid];
    let point = origin + direction * hit.distance;
    let normal = hit.tangent / (hit.tangent | hit.tangent).square_root();
    // The lamp's light falls by the cosine of its direction off the normal, a plane paired with a direction.
    let lit = max(normal & lamp, 0.0) * select(1.0, shadow, first_hit(point + lamp * clearance, lamp).distance < far);
    let parity = abs((floor(solid.across & point) + floor(solid.along & point)) % 2.0);
    // The view ray reflected in the tangent plane, against the lamp: `-d.dual()` is the plane square to d.
    let glint = pow(max(-((normal >> direction).dual() & lamp), 0.0), shininess);
    let colour = solid.colour * (1 - checker * parity) * (ambient + (1 - ambient) * lit) + gloss * glint * lit;
    return Sight(colour, point, normal, solid.reflectivity);
}

fn unit(direction: Point) -> Point {
    return direction / (-(direction.dual() & direction)).square_root();
}

// What the ray through a place on the image sees: along the camera's own -z, turned by its motor.
fn sample(place: vec2<f32>) -> Sample {
    let across = screen(place) * half_view;
    let direction = unit(camera >> (-mv.z.dual() + mv.x.dual() * across.x + mv.y.dual() * across.y));
    let seen = sight(eye, direction);
    // A mirror shows what its reflection sees: the view ray reflected in the tangent plane.
    let reflected = seen.normal >> direction;
    let mirrored = sight(seen.point + reflected * clearance, reflected);
    return mix(seen.colour, mirrored.colour, seen.reflectivity);
}
"""

## 4. The camera's view

A camera is a motor: one at the origin looking along its own -z, backed off, tilted down, turned about the vertical, and carried to the target. The image comes from `supersampled()`, a turned grid of two samples a side in each pixel, averaged; the shader says what one sample sees, `fn sample(place: vec2<f32>) -> Sample`.

In [ ]:
target = mv.point([0.0, 0.2, 0.7])
distance, elevation, half_view = 4.6, np.radians(28.0), 0.42
lamp = mv.direction([0.5, -0.5, 0.7])


def frames(cameras, shape):
    for camera in cameras:
        traced = tracer(shape, quadrics=quadrics, solids=solids, camera=camera, eye=camera >> origin,
                        lamp=lamp, half_view=half_view)["image"]
        yield np.round(np.clip(traced, 0, 1) * 255).astype(np.uint8)


tracer = Shader(source, ga, dict(Point=Point, Plane=Plane, Motor=Motor, Quadric=Quadric))
still, = frames(orbit(target, distance, elevation, np.zeros(1)), (450, 720))
save_figure(image_figure(still), "csg_raytracer");

## 5. Around the scene

The camera circles the target, each frame a fresh trace: the mirror shows the other solids as they come round, and the bowl's cut shows its hollow.

In [ ]:
n_frames = 48
cameras = orbit(target, distance, elevation, np.linspace(0, 2 * np.pi, n_frames, endpoint=False))   # [frames] Motor
show_animation(list(frames(cameras, (330, 528))), "csg_raytracer", 80)

In [ ]:
# checks
# Both families of straight lines through the waist lie in the hyperboloid's surface, and every camera looks at the target.
for direction in (rising, falling):
    np.testing.assert_allclose(evaluate(hyperboloid(waist_axes), waist + direction * reach).kernel, 0, atol=1e-12)
np.testing.assert_allclose(((cameras >> origin) + (cameras >> -mv.z.dual()) * distance - target).kernel, 0, atol=1e-12)